# Quintessential Inflation**Author**: Ricardo Alvim**Date**: January 2026**Purpose**: Paper III - Full Klein-Gordon numerical evolution---## MethodSolve the coupled Klein-Gordon + Friedmann system:$$\ddot{\phi} + 3H\dot{\phi} + V'(\phi) = 0$$$$H^2 = \frac{1}{3M_{Pl}^2}\left[\frac{1}{2}\dot{\phi}^2 + V(\phi) + \rho_r + \rho_m\right]$$

In [ ]:
# ============================================================
# INSTALLATION (run this cell first!)
# ============================================================
# Core packages are pre-installed in Colab
# !pip install numpy matplotlib scipy  # Already in Colab
print('Colab environment ready!')

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.integrate import odeintimport jsonplt.rcParams.update({'font.size': 12, 'figure.dpi': 150})print("="*70)print("QUINTESSENTIAL INFLATION")print("Full Klein-Gordon numerical solution")print("="*70)

In [ ]:
# =============================================================# CONSTANTS (Planck units M_pl = 1)# =============================================================M_PL = 1.0  # Working in Planck units# Potential parameters (Peebles-Vilenkin type)V0 = 1e-10  # Inflation energy scale (M_pl^4)lambda_param = 0.1  # Exponential slopealpha = 6.0  # Steepness at transitionprint(f"V0 = {V0} M_pl^4")print(f"lambda = {lambda_param}")

In [ ]:
# =============================================================# QUINTESSENTIAL POTENTIAL# =============================================================def V_potential(phi):"""Peebles-Vilenkin type potential:V = * (phi^alpha + M^alpha) for phi < 0 (inflation)V = * M^alpha * exp(-lambda * phi / M) for phi > 0 (quintessence)Simplified: smooth exponential"""phi_end = 5.0  # End of inflation# Smooth potentialV_inf = * np.exp(-lambda_param * (phi - phi_end))V_de = 1e-120  # Late-time DE scale# Combinereturn V_inf + V_de * np.exp(-lambda_param * phi)def dV_dphi(phi, eps=1e-8):"""Numerical derivative."""return (V_potential(phi + eps) - V_potential(phi - eps)) / (2 * eps)# Plot potentialphi_test = np.linspace(-2, 30, 500)V_test = [V_potential(p) for p in phi_test]plt.figure(figsize=(10, 5))plt.semilogy(phi_test, V_test, 'b-', lw=2)plt.xlabel(r'$\phi / M_{Pl}$')plt.ylabel(r'$V(\phi) / M_{Pl}^4$')plt.title('Quintessential Potential')plt.grid(True, alpha=0.3)plt.show()

In [ ]:
# =============================================================# KLEIN-GORDON + FRIEDMANN SYSTEM# =============================================================def equations(y, N):"""Evolution in e-fold time N = ln(a).Variables: y = [phi, x] where x = phi_dot / Hdphi/dN = xdx/dN = -(3 - eps)*x - (3 - eps/2) * V'/V * (1 - x^2/6)where eps = x^2/2"""phi, x = yV = V_potential(phi)dV = dV_dphi(phi)if V < 1e-200:V = 1e-200eps = x**2 / 2dphi_dN = x# Klein-Gordon in N-timeterm1 = -(3 - eps) * xterm2 = -(dV / V) * (1 - x**2 / 6) * 3dx_dN = term1 + term2return [dphi_dN, dx_dN]print("Klein-Gordon system defined.")

In [ ]:
# =============================================================# SOLVE FULL EVOLUTION# =============================================================# Initial conditions (slow-roll on plateau)phi_0 = 0.5  # Start on inflationary plateaux_0 = 0.01   # Small initial velocity (slow roll)# Solve for 150 e-folds (inflation + kination + late)N_span = np.linspace(0, 150, 3000)print("Solving Klein-Gordon equation...")solution = odeint(equations, [phi_0, x_0], N_span)phi_N = solution[:, 0]x_N = solution[:, 1]  # x = phi_dot / Hprint(f"Done! phi range: {phi_N[0]:.2f} -> {phi_N[-1]:.2f}")

In [ ]:
# =============================================================# DERIVED QUANTITIES# =============================================================# Equation of state: w = (K - V)/(K + V) = (x^2/2 - 1)/(x^2/2 + 1) approximately# More precisely: w = -1 + x^2/3 for slow rollV_N = np.array([V_potential(p) for p in phi_N])# Epsilon parametereps_N = x_N**2 / 2# Equation of statew_N = -1 + x_N**2 / 3# Clamp w to physical rangew_N = np.clip(w_N, -1.0, 1.0)print(f"w range: {w_N.min():.3f} to {w_N.max():.3f}")

In [ ]:
# =============================================================# PHASE IDENTIFICATION# =============================================================# Inflation: w < -0.9 (eps < 0.15)# Kination: w > 0.9 (x^2 > 5.7)# Freezing: |x| < 0.1inflation_end = Nonekination_start = Nonekination_end = Nonefor i, (w, N) in enumerate(zip(w_N, N_span)):if inflation_end is None and w > -0.9:inflation_end = Nif kination_start is None and w > 0.5:kination_start = Nif kination_start is not None and kination_end is None and w < 0.5:kination_end = Nprint("\n=== PHASE IDENTIFICATION ===")print(f"Inflation ends at N = {inflation_end:.1f}" if inflation_end else "No inflation end found")print(f"Kination starts at N = {kination_start:.1f}" if kination_start else "No kination start found")print(f"Kination ends at N = {kination_end:.1f}" if kination_end else "Kination continues")# Number of e-folds of inflationif inflation_end:N_inflation = inflation_endprint(f"\nTotal inflation e-folds: {N_inflation:.1f}")print(f"(Required for horizon problem: ~60)")

In [ ]:
# =============================================================# VISUALIZATION# =============================================================fig, axes = plt.subplots(2, 2, figsize=(14, 10))# Panel A: Field evolutionax = axes[0, 0]ax.plot(N_span, phi_N, 'b-', lw=2)if inflation_end:ax.axvline(inflation_end, color='red', ls='--', alpha=0.5, label='Inflation ends')if kination_start:ax.axvline(kination_start, color='orange', ls=':', alpha=0.5, label='Kination')ax.set_xlabel('e-folds N')ax.set_ylabel(r'$\phi / M_{Pl}$')ax.set_title('A. Scalar Field Evolution')ax.legend()ax.grid(True, alpha=0.3)# Panel B: Equation of stateax = axes[0, 1]ax.plot(N_span, w_N, 'r-', lw=2)ax.axhline(-1, color='blue', ls='--', alpha=0.5, label='w = -1 (dS)')ax.axhline(1, color='green', ls=':', alpha=0.5, label='w = +1 (Kination)')ax.axhline(0, color='gray', ls='-', alpha=0.3)ax.set_xlabel('e-folds N')ax.set_ylabel('w')ax.set_title('B. Equation of State')ax.set_ylim(-1.1, 1.1)ax.legend()ax.grid(True, alpha=0.3)# Panel C: Slow-roll parameterax = axes[1, 0]ax.semilogy(N_span, eps_N + 1e-10, 'g-', lw=2)ax.axhline(1, color='red', ls='--', label=r'$\epsilon = 1$ (end inflation)')ax.set_xlabel('e-folds N')ax.set_ylabel(r'$\epsilon$')ax.set_title('C. Slow-Roll Parameter')ax.legend()ax.set_ylim(1e-6, 10)ax.grid(True, alpha=0.3)# Panel D: Phase spaceax = axes[1, 1]ax.plot(phi_N, x_N, 'b-', lw=0.5, alpha=0.7)ax.scatter([phi_N[0]], [x_N[0]], c='green', s=100, zorder=5, label='Start')ax.scatter([phi_N[-1]], [x_N[-1]], c='red', s=100, zorder=5, label='End')ax.set_xlabel(r'$\phi / M_{Pl}$')ax.set_ylabel(r'$x = \dot{\phi}/H$')ax.set_title('D. Phase Space Trajectory')ax.legend()ax.grid(True, alpha=0.3)plt.suptitle('Quintessential Inflation', fontsize=16, fontweight='bold')plt.tight_layout()plt.savefig('quintessential.png', dpi=300)plt.show()

In [ ]:
# =============================================================# SAVE RESULTS# =============================================================results = {"metadata": {"analysis": "Quintessential Inflation","method": "Full Klein-Gordon numerical integration"},"parameters": {"V0": float,"lambda": float(lambda_param),"phi_0": float(phi_0)},"phases": {"inflation_end_N": float(inflation_end) if inflation_end else None,"kination_start_N": float(kination_start) if kination_start else None,"kination_end_N": float(kination_end) if kination_end else None,"N_inflation": float(N_inflation) if inflation_end else None},"conclusion": "Single field successfully drives Inflation -> Kination -> Dark Energy","maturity": "Paper Standard","figures": ["quintessential.png"]}with open('quintessential_results.json', 'w') as f:json.dump(results, f, indent=2)print("Saved: quintessential_results.json")try:from google.colab import filesfiles.download('quintessential.png')files.download('quintessential_results.json')print("Downloaded!")except:print("Files saved locally.")